In [1]:
from func import *

# Read data

In [2]:
train_df = pd.read_csv("processed5/train_data.csv")
test_df = pd.read_csv("processed5/test_data.csv")

train_df["logClosePrice"] = np.log1p(train_df["ClosePrice"])
train_df.drop(columns=["ClosePrice"], inplace=True)
test_df["logClosePrice"] = np.log1p(test_df["ClosePrice"])
test_df.drop(columns=["ClosePrice"], inplace=True)

# Elastic Net

In [3]:
from sklearn.linear_model import ElasticNet

In [4]:

# Tuning
en_param_grid = param_grid = {
    "model__alpha": np.logspace(-4, 2, 25),
    "model__l1_ratio": np.linspace(0.05, 0.95, 19),
}

res = grid_tune_with_make_model_pipeline(
    train_df=train_df,
    target_col="logClosePrice",
    model=ElasticNet(max_iter=20000),
    param_grid=param_grid,
    col_drop_list=["ClosePrice"],
    card_threshold=20,
    scoring="r2",
    cv=5
)

print(res["best_score"], res["best_params"])

Fitting 5 folds for each of 475 candidates, totalling 2375 fits
0.8627505332713833 {'model__alpha': 0.01778279410038923, 'model__l1_ratio': 0.05}


In [4]:
alpha = 0.01778279410038923
l_1_ratio = 0.05
model = ElasticNet(alpha=alpha,
                   l1_ratio=l_1_ratio)

elastic_net_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)
el_metrics = elastic_net_result["Metrics"]
print(el_metrics)

{'R2(log)': 0.8631442453920551, 'R2': 0.7552260479068467, 'MAPE': 17.16603363104206, 'MdAPE': 12.15938822102785, 'RMSE': 434158.57416388416, 'MAE': 200882.27025619644, 'Bias(mean residual)': -27121.925872513202, 'APE_95pct': 48.87833389931527, 'APE_99pct': 85.6371301137496, 'APE_max': 246.12250152637475, 'Train_R2(log)': 0.8992553084461341, 'Test_R2(log)': 0.8631442453920551, 'R2_gap': 0.03611106305407896}


# Random Forest

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [6]:
X_train = train_df.drop(columns=["logClosePrice"], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()


In [3]:

rf_param_dist = {
    "model__n_estimators": randint(300, 1000),
    "model__max_depth": [None] + list(range(10, 60, 10)),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "model__min_samples_leaf": randint(1, 15),
    "model__min_samples_split": randint(2, 20)
}

model = make_model_pipeline(model=RandomForestRegressor(
    n_jobs=1,
    random_state=42),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)


rs = RandomizedSearchCV(
    model,
    param_distributions=rf_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(rs.best_params_)

{'model__max_depth': 40, 'model__max_features': 'log2', 'model__min_samples_leaf': 8, 'model__min_samples_split': 13, 'model__n_estimators': 713}


In [7]:
n_estimators = 713
max_depth = 40
max_features = 'log2'
min_samples_leaf = 8
min_samples_split = 13

rf_model = RandomForestRegressor(n_estimators=n_estimators,
                                 max_depth=max_depth,
                                 max_features=max_features,
                                 min_samples_leaf=min_samples_leaf,
                                 min_samples_split=min_samples_split)

rf_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=rf_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)

rf_metrics = rf_result["Metrics"]
print(rf_metrics)

{'R2(log)': 0.7050520320917828, 'R2': 0.4911035907632221, 'MAPE': 24.542421644566765, 'MdAPE': 18.348638493516678, 'RMSE': 626008.85375524, 'MAE': 298658.5197551141, 'Bias(mean residual)': -147966.55668011564, 'APE_95pct': 66.5353717343296, 'APE_99pct': 115.4064870192664, 'APE_max': 287.67208747763175, 'Train_R2(log)': 0.969197820511347, 'Test_R2(log)': 0.7050520320917828, 'R2_gap': 0.26414578841956415}


# XGB

In [8]:
from xgboost import XGBRegressor
from scipy.stats import randint, uniform, loguniform

In [8]:

xgb_param_dist = {
    "model__max_depth": randint(3, 7),                    # 3–6
    "model__min_child_weight": randint(8, 31),           # 8–30
    "model__learning_rate": loguniform(0.01, 0.08),      # small LR
    "model__n_estimators": randint(400, 1400),
    "model__subsample": uniform(0.6, 0.3),               # 0.6–0.9
    "model__colsample_bytree": uniform(0.5, 0.4),        # 0.5–0.9
    "model__reg_lambda": loguniform(1.0, 50.0),          # strong L2
    "model__reg_alpha": loguniform(1e-4, 2.0),           # mild–moderate L1
}

model = make_model_numeric_only_pipeline(model=XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=1),
 num_cols=num_cols,
num_scaler=None)


xgb_rs = RandomizedSearchCV(
    model,
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

xgb_rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(xgb_rs.best_params_)

{'model__colsample_bytree': 0.7844598129752072, 'model__learning_rate': 0.05383345943137039, 'model__max_depth': 6, 'model__min_child_weight': 14, 'model__n_estimators': 1219, 'model__reg_alpha': 1.10973369349242, 'model__reg_lambda': 4.736558858366902, 'model__subsample': 0.755325405158244}


In [9]:
xgb_model = XGBRegressor(n_estimators=1219,
                         learning_rate=0.05383345943137039,
                         max_depth=6,
                         subsample=0.755325405158244,
                         colsample_bytree=0.7844598129752072,
                         min_child_weight=14,
                         reg_lambda=4.736558858366902,
                         reg_alpha=1.10973369349242)

# xgb_model = XGBRegressor(n_estimators=xgb_rs.best_params_["model__n_estimators"],
#                          learning_rate=xgb_rs.best_params_["model__learning_rate"],
#                          max_depth=xgb_rs.best_params_["model__max_depth"],
#                          subsample=xgb_rs.best_params_["model__subsample"],
#                          colsample_bytree=xgb_rs.best_params_["model__colsample_bytree"],
#                          min_child_weight=xgb_rs.best_params_["model__min_child_weight"],
#                          reg_lambda=xgb_rs.best_params_["model__reg_lambda"],
#                          reg_alpha=xgb_rs.best_params_["model__reg_alpha"]
#                          )

xgb_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=xgb_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20,
    num_scaler=None,
    num_only=True
)

xgb_metrics = xgb_result["Metrics"]
print(xgb_metrics)

{'R2(log)': 0.9224351015790591, 'R2': 0.8763345621685927, 'MAPE': 12.183347241653113, 'MdAPE': 8.290487114820188, 'RMSE': 308595.75808726, 'MAE': 146815.4192791814, 'Bias(mean residual)': 1540.9456341720306, 'APE_95pct': 36.403031809701524, 'APE_99pct': 62.463311746031664, 'APE_max': 199.30575000000024, 'Train_R2(log)': 0.953032013802761, 'Test_R2(log)': 0.9224351015790591, 'R2_gap': 0.030596912223701977}


# Stacking

In [10]:
from sklearn.ensemble import StackingRegressor


en_full = make_model_pipeline(
    model=ElasticNet(alpha=0.01778279410038923,
                     l1_ratio=0.05),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)

xgb_num = make_model_numeric_only_pipeline(
    model=XGBRegressor(n_estimators=1219,
                       learning_rate=0.05383345943137039,
                       max_depth=6,
                       subsample=0.755325405158244,
                       colsample_bytree=0.7844598129752072,
                       min_child_weight=14,
                       reg_lambda=4.736558858366902,
                       reg_alpha=1.10973369349242),
    num_cols=num_cols,
    num_scaler=None
)

# ---------- Stack them ----------
stack = StackingRegressor(
    estimators=[
        ("en_full", en_full),
        ("xgb_num", xgb_num),
    ],
    final_estimator=XGBRegressor(
        max_depth=2,
        n_estimators=200,
        learning_rate=0.05
    ),
    cv=5,
    n_jobs=-1
)

# Fit on log target
y_train = train_df["logClosePrice"]
stack.fit(X_train, y_train)

# Predict log target
X_test = test_df.drop("logClosePrice", axis=1)
y_test = test_df["logClosePrice"]
y_train_pred = stack.predict(X_train)
y_pred = stack.predict(X_test)

stack_metrics = compute_metrics(y_test, y_pred, y_train, y_train_pred)

In [11]:
stack_metrics

{'R2(log)': 0.9253911611150452,
 'R2': 0.880246962599021,
 'MAPE': 11.89597460229059,
 'MdAPE': 8.098419198292008,
 'RMSE': 303675.0076576741,
 'MAE': 143567.5296413145,
 'Bias(mean residual)': 1934.0104060418257,
 'APE_95pct': 35.968231839225645,
 'APE_99pct': 62.16171149053832,
 'APE_max': 212.02678571428595,
 'Train_R2(log)': 0.953137998963144,
 'Test_R2(log)': 0.9253911611150452,
 'R2_gap': 0.027746837848098838}

# Residual model

In [12]:
residual_model = XGBRegressor(
    max_depth=3,
    learning_rate=0.03,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
)

residual = y_train - y_train_pred

residual_model.fit(X_train[num_cols], residual)

pred1 = stack.predict(X_test)
pred2 = residual_model.predict(X_test[num_cols])

train_pred1 = stack.predict(X_train)
train_pred2 = residual_model.predict(X_train[num_cols])

y_residual_pred = pred1 + pred2
y_train_residual_pred = train_pred1 + train_pred2

In [13]:
residual_metrics = compute_metrics(y_test, y_residual_pred, y_train, y_train_residual_pred)

In [14]:
residual_metrics

{'R2(log)': 0.9267660256598371,
 'R2': 0.8805756278193271,
 'MAPE': 11.721959686462318,
 'MdAPE': 7.8927662671743395,
 'RMSE': 303257.9994941038,
 'MAE': 141463.69679694006,
 'Bias(mean residual)': -6664.498961152179,
 'APE_95pct': 35.423015033956744,
 'APE_99pct': 61.78747229452061,
 'APE_max': 204.30335714285738,
 'Train_R2(log)': 0.9550838075809993,
 'Test_R2(log)': 0.9267660256598371,
 'R2_gap': 0.02831778192116219}

# Summary

In [15]:
models_summary = {
    "Elastic Net": el_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics,
    "StackingRegressor": stack_metrics,
    "Residual Model": residual_metrics,
}

df = pd.DataFrame.from_dict(models_summary).transpose()
df

,R2(log),R2,MAPE,MdAPE,RMSE,MAE,Bias(mean residual),APE_95pct,APE_99pct,APE_max,Train_R2(log),Test_R2(log),R2_gap
Elastic Net,0.863144,0.755226,17.166034,12.159388,434158.574164,200882.270256,-27121.925873,48.878334,85.637130,246.122502,0.899255,0.863144,0.036111
Random Forest,0.705052,0.491104,24.542422,18.348638,626008.853755,298658.519755,-147966.556680,66.535372,115.406487,287.672087,0.969198,0.705052,0.264146
XGBoost,0.922435,0.876335,12.183347,8.290487,308595.758087,146815.419279,1540.945634,36.403032,62.463312,199.305750,0.953032,0.922435,0.030597
StackingRegressor,0.925391,0.880247,11.895975,8.098419,303675.007658,143567.529641,1934.010406,35.968232,62.161711,212.026786,0.953138,0.925391,0.027747
Residual Model,0.926766,0.880576,11.721960,7.892766,303257.999494,141463.696797,-6664.498961,35.423015,61.787472,204.303357,0.955084,0.926766,0.028318
